# Week 3: Naive Bayes Classification

# First time here?



1.   Copy this file to your Google Drive so you can edit it: **File->Save a copy in Drive** (requires a personal Google account)
2.   Done? Consider giving your copy a better name (e.g., including your name and/or student ID) - just click on the filename in the top left and edit it.
3.   (Want to re-open your copy in future? Go: **File->Open notebook** and select the Google Drive tab)
4.   (Want to find your file on your Google Drive? Google puts Colab files in a folder called "Colab Notebooks")

# Our implementations (NB)

In this notebook we're going to focus on understanding a very simple Python implementation of Naive Bayes (NB).

It should be useful for checking your understanding from your research this week, and it may also be useful to return to and extend in future.

## `MyNaiveBayesClassifier`

Below is a really simple NumPy implementation of a Naive Bayes (NB) classifier. We'll take some time to look at it in detail in a moment, but for now let's just run it and see how it performs.

Click the play button on the code cell to define the class, and then scroll down to the next text cell and keep reading:

In [1]:
import numpy as np

class MyNaiveBayesClassifier:

  # The class constructor
  def __init__(self):
    # nothing to do - Scikit-learn doesn't support any interesting hyperparameters, and neither will we
    pass

  # The method we call to fit the model to training data
  def fit(self, train_examples, train_labels):

    # calculate all the numbers we'll need for our Bayes theorem calculations during prediction

    # what are the unique labels in this problem
    self.labels = np.unique(train_labels)
    # how many are there
    self.n_labels = self.labels.shape[0]
    # how many predictive features are there
    self.n_features = train_examples.shape[1]

    # arrays to store our data (I'll use one _row_ for information about each label; I could equally have used one column)
    self.prior = np.empty(self.n_labels, dtype=float)
    self.means = np.empty((self.n_labels, train_examples.shape[1]), dtype=float)
    self.stds = np.empty((self.n_labels, train_examples.shape[1]), dtype=float)

    # for each unique label, calculate and store its prior probability, and the mean and standard deviation of its associated examples
    for i in range(self.n_labels):
      # how frequently did this label appear in the training data
      self.prior[i] = np.sum(train_labels==self.labels[i]) / train_labels.shape[0]
      # what was the mean of its associated training examples
      self.means[i,:] = np.mean(train_examples[train_labels==self.labels[i],:], axis=0)
      # what was the standard deviation of its associated training examples
      self.stds[i,:] = np.std(train_examples[train_labels==self.labels[i],:], axis=0)

  # The method we call to generate predictions for a batch of testing examples (exactly the same as last time)
  def predict(self, test_examples):

    # how many test examples are there
    n_test_examples = test_examples.shape[0]

    # make numpy array the right size to store our predicted labels
    predicted_labels = np.empty(n_test_examples, dtype=self.labels.dtype)

    # loop over all the testing examples that were passed to us
    for i in range(n_test_examples):

        # For this one testing example...
        this_example = test_examples[i,:]

        # generate a prediction and store it
        predicted_labels[i] = self.predict_one(this_example)

    # return all the predictions
    return predicted_labels

  # A method to generate a prediction for a single example
  def predict_one(self, this_example):

    # for each label in the problem, its posterior probability is proportional to: (prior probability of seeing that label) * (likelihood of seeing these feature values in an observation with that label)
    # we'll calculate these scores for each label, and whichever label has the largest score becomes our prediction
    # (we calculated all the numbers we need for these calculations during fitting - we're using Normal distributions to model the likelihoods)

    # an array to store our scores
    probabilities_ = np.zeros(self.n_labels, dtype=float)

    # for each unique label, calculate prior * likelihood
    for i in range(self.n_labels):
      # start with the prior
      probabilities_[i] = self.prior[i]
      # then multiply by ...
      for j in range(self.n_features):
        # to get the overall likelihood of the whole example: we treat the individual feature values we saw as _independent events_ given the label, and multiply their individual likelihoods together
        probabilities_[i] *= self.normal_pdf_score(self.means[i,j], self.stds[i,j], this_example[j])
        # (this is NB's "class-conditional independence assumtion" - which is quite a "naive" assumption; as in, it's unlikely to hold true in reality - but NB often still gives very good results...)

    # find the maximum score - which label won?
    winner_index = np.argmax(probabilities_)

    # return the corresponding label as our prediction
    return self.labels[winner_index]

  def normal_pdf_score(self, mean, std, x):

    # return the probability density of a particular Normal distribution (mean, std) at a particular point (x)
    return ( 1 / (np.sqrt(2*np.pi) * std) ) * np.exp( (-np.pow(x-mean, 2)) / (2*np.pow(std,2)) )



## Basic supervised learning recipe

The code below shows our basic supervised learning recipe again. We're using it to train and evaluate on the iris.csv dataset again, but this time we're using the basic NB implementation we defined above.

*   How accurate is the model? ** 95 %**

In [6]:
# Import the packages we use
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report


# Load all the observations from file
observations = pd.read_csv('https://helward.mmu.ac.uk/STAFF/J.Darby/ML/iris.csv')

# Set the name of the target feature
target_feature = 'species'

# Shuffle all the observations
observations_shuffled = observations.sample(frac=1, random_state=99)

# Set the fraction of observations we will use for testing
testing_fraction = 0.4
split_index = int(observations_shuffled.shape[0] * testing_fraction)

# Split into testing observations and training observations ("horizontal split")
observations_test = observations_shuffled.iloc[:split_index]
observations_train = observations_shuffled.iloc[split_index:]

# Split into testing examples and testing labels ("vertical split")
test_examples = observations_test.drop(columns=target_feature).to_numpy()
test_labels = observations_test[target_feature].to_numpy()

# Split into training examples and training labels ("vertical split")
train_examples = observations_train.drop(columns=target_feature).to_numpy()
train_labels = observations_train[target_feature].to_numpy()

# Create a NB classification model
model = MyNaiveBayesClassifier()

# Call the model's fitting/training algorithm, passing in our training examples and training labels
model.fit(train_examples, train_labels)

# Use the trained model to generate predictions based on our testing examples
predictions = model.predict(test_examples)

# Find the total number of model predictions that matched with their corresponding testing labels
correct_predictions = sum(predictions == test_labels)
# Calculate the model's accuracy: the fraction of predictions that were correct
accuracy = correct_predictions / len(test_labels)
# Display the accuracy as a single quantitative measure of overall performance
print("Accuracy:", accuracy, "(or", round(accuracy*100, 1), "%)")

#  Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, predictions))

#  Classification report
print("\nClassification Report:")
print(classification_report(test_labels, predictions))

Accuracy: 0.95 (or 95.0 %)

Confusion Matrix:
[[17  0  0]
 [ 0 20  2]
 [ 0  1 20]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        17
  versicolor       0.95      0.91      0.93        22
   virginica       0.91      0.95      0.93        21

    accuracy                           0.95        60
   macro avg       0.95      0.95      0.95        60
weighted avg       0.95      0.95      0.95        60



# Review

Now take some time to have a careful look through the `MyNaiveBayesClassifier` class. Can you match each of the stages up with your research from this week? Naive Bayes, though a simple / naive model, can be quite conceptually difficult, so some comments have been left in to help you. Take time to read over them / add to them / adjust them as you need - and check you can follow what is happening at each stage of classification:

*    **review the code and comments and make sure you're happy with how the model works**

**Tip**: consider adding print statements to `MyNaiveBayesClassifier` to help you understand what's happening. Re-run the code cell that defines the class before re-running the recipe code to see the output. Remember about printing things like `.shape` as well as printing out the raw contents of arrays (which can produce many lines of output).

**Tip**: the only thing that's new in terms of NumPy is the `axis=` argument, which lets us ask NumPy functions to operate along a particular dimension of a multidimensional NumPy array. For example, we use `axis=0` to operate _down columns_ of a 2D array when computing the mean/standard deviation (i.e., we're operating on individual features). (`axis=1` would have operated along rows - not what we want - and not specifying `axis=` at all would have calculated a mean/standard deviation for _all values_ in the whole 2D array.

## Next steps

You could, if you wanted to, add more code to this `MyNaiveBayesClassifier` class in order to gain further insight into the workings of the model's `fit()` and `predict()` methods. For example, see how we use the Matplotlib package for visualisation in the second notebook.